In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
stg_transactions = spark.read.table("stg_transactions")
stg_daily_account_balance = spark.read.table("stg_daily_account_balance")
stg_loan_payments = spark.read.table("stg_loan_payments")

StatementMeta(, 51c62fbc-e37e-4ff3-8d6b-eb338af3dc22, 3, Finished, Available, Finished, False)

In [7]:
fact_transactions = stg_transactions.select(
    "transaction_id",
    "date_key",
    "account_key",
    "customer_key",
    "branch_key",
    "product_key",
    "channel_key",
    "transaction_amount",
    "transaction_fee",
    "balance_after_transaction"
)

fact_transactions.write.mode("overwrite").format("delta").saveAsTable("fact_transactions")

StatementMeta(, 51c62fbc-e37e-4ff3-8d6b-eb338af3dc22, 9, Finished, Available, Finished, False)

In [9]:
fact_daily_account_balance = stg_daily_account_balance.select(
    "date_key",
    "account_key",
    "customer_key",
    "branch_key",
    "product_key",
    "opening_balance",
    "debit_amount",
    "credit_amount",
    "closing_balance",
    "transaction_count"
)

fact_daily_account_balance.write.mode("overwrite").format("delta").saveAsTable("fact_daily_account_balance")

StatementMeta(, 51c62fbc-e37e-4ff3-8d6b-eb338af3dc22, 11, Finished, Available, Finished, False)

In [10]:
fact_loan_payments = stg_loan_payments.select(
    "loan_payment_id",
    "date_key",
    "account_key",
    "customer_key",
    "branch_key",
    "product_key",
    "emi_amount",
    "principal_amount",
    "interest_amount",
    "outstanding_principal",
    "days_past_due"
)

fact_loan_payments.write.mode("overwrite").format("delta").saveAsTable("fact_loan_payments")

StatementMeta(, 51c62fbc-e37e-4ff3-8d6b-eb338af3dc22, 12, Finished, Available, Finished, False)

In [ ]:
display(spark.sql("SHOW TABLES"))
display(spark.read.table("fact_transactions"))
display(spark.read.table("fact_daily_account_balance"))
display(spark.read.table("fact_loan_payments"))

StatementMeta(, 51c62fbc-e37e-4ff3-8d6b-eb338af3dc22, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5d5ef63e-2707-4373-96c1-a4ebacb160c5)

In [2]:
%%sql

CREATE OR REPLACE TABLE fact_transactions_final AS
SELECT
    s.transaction_id,
    CAST(s.date_key AS STRING)      AS date_key,
    CAST(s.account_key AS STRING)   AS account_key,
    CAST(s.customer_key AS STRING)  AS customer_key,
    CAST(s.branch_key AS STRING)    AS branch_key,
    CAST(s.product_key AS STRING)   AS product_key,
    CAST(s.channel_key AS STRING)   AS channel_key,

    s.transaction_amount,
    s.transaction_fee,
    s.balance_after_transaction

FROM stg_transactions s
LEFT JOIN dim_customer c
    ON s.customer_key = c.customer_key
LEFT JOIN dim_account a
    ON s.account_key = a.account_key
LEFT JOIN dim_branch b
    ON s.branch_key = b.branch_key
LEFT JOIN dim_product p
    ON s.product_key = p.product_key
LEFT JOIN dim_channel ch
    ON s.channel_key = ch.channel_key
LEFT JOIN dim_date d
    ON s.date_key = d.date_key;

StatementMeta(, 7f027ab6-0309-4e4e-8c92-2473d91a2545, 3, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [3]:
%%sql

CREATE OR REPLACE TABLE fact_daily_account_balance_final AS
SELECT
    CAST(s.date_key AS STRING)      AS date_key,
    CAST(s.account_key AS STRING)   AS account_key,
    CAST(s.customer_key AS STRING)  AS customer_key,
    CAST(s.branch_key AS STRING)    AS branch_key,
    CAST(s.product_key AS STRING)   AS product_key,

    s.opening_balance,
    s.debit_amount,
    s.credit_amount,
    s.closing_balance,
    s.transaction_count

FROM stg_daily_account_balance s
LEFT JOIN dim_customer c
    ON s.customer_key = c.customer_key
LEFT JOIN dim_account a
    ON s.account_key = a.account_key
LEFT JOIN dim_branch b
    ON s.branch_key = b.branch_key
LEFT JOIN dim_product p
    ON s.product_key = p.product_key
LEFT JOIN dim_date d
    ON s.date_key = d.date_key;

StatementMeta(, 7f027ab6-0309-4e4e-8c92-2473d91a2545, 4, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [ ]:
%%sql

CREATE OR REPLACE TABLE fact_loan_payments_final AS
SELECT
    s.loan_payment_id,
    CAST(s.date_key AS STRING)      AS date_key,
    CAST(s.account_key AS STRING)   AS account_key,
    CAST(s.customer_key AS STRING)  AS customer_key,
    CAST(s.branch_key AS STRING)    AS branch_key,
    CAST(s.product_key AS STRING)   AS product_key,

    s.emi_amount,
    s.principal_amount,
    s.interest_amount,
    s.outstanding_principal,
    s.days_past_due

FROM stg_loan_payments s
LEFT JOIN dim_customer c
    ON s.customer_key = c.customer_key
LEFT JOIN dim_account a
    ON s.account_key = a.account_key
LEFT JOIN dim_branch b
    ON s.branch_key = b.branch_key
LEFT JOIN dim_product p
    ON s.product_key = p.product_key
LEFT JOIN dim_date d
    ON s.date_key = d.date_key;

StatementMeta(, 7f027ab6-0309-4e4e-8c92-2473d91a2545, 5, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [7]:
fact_daily_account_balance_final = spark.read.table("fact_daily_account_balance_final")
fact_daily_account_balance_final.printSchema()

StatementMeta(, 7f027ab6-0309-4e4e-8c92-2473d91a2545, 9, Finished, Available, Finished, False)

root
 |-- date_key: string (nullable = true)
 |-- account_key: string (nullable = true)
 |-- customer_key: string (nullable = true)
 |-- branch_key: string (nullable = true)
 |-- product_key: string (nullable = true)
 |-- opening_balance: double (nullable = true)
 |-- debit_amount: double (nullable = true)
 |-- credit_amount: double (nullable = true)
 |-- closing_balance: double (nullable = true)
 |-- transaction_count: long (nullable = true)



In [9]:
customer_table = spark.read.table("dim_customer")
customer_table.printSchema()

StatementMeta(, 7f027ab6-0309-4e4e-8c92-2473d91a2545, 11, Finished, Available, Finished, False)

root
 |-- customer_key: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- full_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- date_of_birth: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- kyc_status: string (nullable = true)
 |-- onboard_date: string (nullable = true)



In [ ]:
t